# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
#Libraries
import pandas as pd
import numpy as np  
from scipy.stats import chi2_contingency
import statsmodels.api as sm


from matplotlib import pyplot as plt 
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

####  HomePlanet vs Transported 


In [ ]:

# -- > Passengers from Europa were more likely to be Transported.
# --> passengers from Earth were more likely NOT to be Transported.
# --> Passengers from Mars had about a 50/50 chance of being Transported.
plt.figure(figsize=(8,5))
sns.countplot(data=spaceship, x="HomePlanet", hue="Transported")
plt.title("Counts of Transported by HomePlanet")
plt.show()


# Chi-Square Test  --- statistically correlated 
# Null hypothesis (H₀): HomePlanet and Transported are independent (no relationship).
# Alternative hypothesis (H₁): They are associated (dependent).
ct = pd.crosstab(spaceship["HomePlanet"], spaceship["Transported"])
res = chi2_contingency(ct)
res.pvalue


#### CryoSleep vs Transported

In [ ]:
# grouped bar chart
plt.figure(figsize=(8,5))
sns.countplot(data=spaceship, x="CryoSleep", hue="Transported")
plt.title("Counts of Transported by CryoSleep")
plt.show()


# Chi-Square Test  --- statistically correlated 
# Null hypothesis (H₀): CryoSleep and Transported are independent (no relationship).
# Alternative hypothesis (H₁): They are associated (dependent).
ct1 = pd.crosstab(spaceship["CryoSleep"], spaceship["Transported"])
res1 = chi2_contingency(ct1)
res1.pvalue

#### VIP vs Transported

In [ ]:
# grouped bar chart
plt.figure(figsize=(8,5))
sns.countplot(data=spaceship, x="VIP", hue="Transported")
plt.title("Counts of Transported by VIP")
plt.show()


# Chi-Square Test  --- statistically correlated 
# Null hypothesis (H₀): VIP and Transported are independent (no relationship).
# Alternative hypothesis (H₁): They are associated (dependent).
ct2 = pd.crosstab(spaceship["VIP"], spaceship["Transported"])
res2 = chi2_contingency(ct2)
res2.pvalue

#### Age VS Transported



In [ ]:
# grouped histogram
sns.histplot(spaceship,x='Age',hue='Transported',bins = 20,fill=False)
plt.show()

# two-sample t-test to test Do the mean ages differ between Transported and Non-Transported? 

# logistic regression with age as the predictor to see if age associated with the binary target in a predictive sense?
# coefficient for age = -0.0017,for each +1 increase in Age
# the log-odds of being Transported=1 decreases by 0.0017.
# p-value = 0.012 ,Conclusion: Age is statistically associated with the transport outcome.
X = spaceship.loc[spaceship['Age'] > 0, ['Age']]   
y = spaceship.loc[spaceship['Age'] > 0, 'Transported']  
model = sm.Logit(y, X).fit()
model.summary()



#### RoomService VS Transported


In [ ]:
X1 = spaceship.loc[spaceship['RoomService'] > 0, ['RoomService']]   
y1 = spaceship.loc[spaceship['RoomService'] > 0, 'Transported']  
model1 = sm.Logit(y1, X1).fit()
model1.summary()

#### FoodCourt vs Transported

In [ ]:
# correlated
X2 = spaceship.loc[spaceship['FoodCourt'] > 0, ['FoodCourt']]   
y2 = spaceship.loc[spaceship['FoodCourt'] > 0, 'Transported']  
model2 = sm.Logit(y2, X2).fit()
model2.summary()

#### ShoppingMall  vs Transported

In [ ]:
# p-value = 0.307, not correlated
X3 = spaceship.loc[spaceship['ShoppingMall'] > 0, ['ShoppingMall']]   
y3 = spaceship.loc[spaceship['ShoppingMall'] > 0, 'Transported']  
model3 = sm.Logit(y3, X3).fit()
model3.summary()

#### Spa vs Transported

In [ ]:
# correlated
X4 = spaceship.loc[spaceship['Spa'] > 0, ['Spa']]   
y4 = spaceship.loc[spaceship['Spa'] > 0, 'Transported']  
model4 = sm.Logit(y4, X4).fit()
model4.summary()

#### VRDeck vs Transported

In [ ]:
# correlated
X5 = spaceship.loc[spaceship['VRDeck'] > 0, ['VRDeck']]   
y5 = spaceship.loc[spaceship['VRDeck'] > 0, 'Transported']  
model5 = sm.Logit(y5, X5).fit()
model5.summary()

####  train test split

In [ ]:

features_df = spaceship[spaceship.columns.drop(['PassengerId','Destination','Name','ShoppingMall'])]
target_df = spaceship['Transported']
X_train,X_test,y_train,y_test = train_test_split(features_df,target_df,test_size=0.2,random_state=42)
X_train.shape,X_test.shape
X_train
X_test


#### create cabin_deck column 

In [ ]:
def create_cabin_deck(train):
    train['Cabin_deck'] = train['Cabin'].str[0]
    train =train.drop(columns = 'Cabin')
    return train

X_train = create_cabin_deck(X_train)
X_test = create_cabin_deck(X_test)


#### data processing  -- imputing 


In [ ]:

def imputer(train):

    numeric_cols = train.select_dtypes(include='number').columns
    categorical_cols = train.select_dtypes(include=["object", "category"]).columns

    train_numeric = train[numeric_cols]
    train_categorical = train[categorical_cols]

    imputer_numeric= SimpleImputer(strategy='median')
    imputer_catogorical = SimpleImputer(strategy='most_frequent')

    imputer_numeric.fit(train_numeric)
    imputer_catogorical.fit(train_categorical)

    train_numeric_imputed = imputer_numeric.transform(train_numeric)
    train_numeric_imputed_df = pd.DataFrame(train_numeric_imputed,columns = imputer_numeric.get_feature_names_out(),index = train.index)
    
    train_catogorical_imputed = imputer_catogorical.transform(train_categorical)
    train_catogorical_imputed_df = pd.DataFrame(train_catogorical_imputed,columns = imputer_catogorical.get_feature_names_out(),index = train.index)
    
    train_final = pd.concat([train_numeric_imputed_df,train_catogorical_imputed_df],axis=1)
    return train_final,imputer_numeric,imputer_catogorical,numeric_cols,categorical_cols

In [ ]:
train_final,imputer_numeric,imputer_catogorical,numeric_cols,categorical_cols = imputer(X_train)

# apply imputer to test 
test_numeric = X_test[numeric_cols]
test_numeric_imputed = imputer_numeric.transform(test_numeric)
test_numeric_imputed_df = pd.DataFrame(test_numeric_imputed,columns = imputer_numeric.get_feature_names_out(),index = X_test.index)
    
test_categorical = X_test[categorical_cols]
test_categorical_imputed = imputer_catogorical.transform(test_categorical)
test_categorical_imputed_df = pd.DataFrame(test_categorical_imputed,columns = imputer_catogorical.get_feature_names_out(),index = X_test.index)

test_final = pd.concat([test_numeric_imputed_df,test_categorical_imputed_df],axis=1)

#### data processing - encoder

In [ ]:

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_final_encoded = encoder.fit_transform(train_final[categorical_cols])   

encoder_cols = encoder.get_feature_names_out()          
train_final_encoded = pd.DataFrame(train_final_encoded, columns=encoder_cols, index=train_final.index)
train_final_encoded_df = pd.concat([train_final_encoded,train_final[numeric_cols]],axis=1)
train_final_encoded_df


In [ ]:
# use trained encoder , transform test data 
test_final_encoded = encoder.transform(test_final[categorical_cols])   
encoder_cols = encoder.get_feature_names_out()          
test_final_encoded = pd.DataFrame(test_final_encoded, columns=encoder_cols, index=test_final.index)
test_final_encoded_df = pd.concat([test_final_encoded,test_final[numeric_cols]],axis=1)
test_final_encoded_df

#### Run models

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
xgb_model = XGBClassifier(n_estimators = 400, max_depth = 5)
xgb_model.fit(train_final_encoded_df,y_train)

- Evaluate your model

In [ ]:
pred = xgb_model.predict(test_final_encoded_df)

print(f"Classification Report:","\n", classification_report(y_test,pred))


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
from sklearn.model_selection import GridSearchCV

- Run Grid Search

In [ ]:
grid = {"n_estimators" : [300, 400, 500, 600, 750],
        'learning_rate': [0.03, 0.1, 0.2],
        "max_depth": [5, 20, 30, 40]}

grid_model = GridSearchCV(estimator = xgb_model,
                     param_grid = grid,
                     cv = 5,
                     verbose = 10,
                     n_jobs=-1)

grid_model.fit(train_final_encoded_df,y_train)



In [ ]:
grid_model.best_params_

- Evaluate your model

In [238]:
best_model = grid_model.best_estimator_
print(best_model)

prediction = best_model.predict(test_final_encoded_df)
print(f"Classification Report:","\n", classification_report(y_test,prediction))


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)
Classification Report: 
               precision    recall  f1-score   support

       False       0.81      0.73      0.77       861
        True       0.76      0.83      0.80       878

    accuracy                           0.78      1739
